## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [1]:
# loading text/data from a file(pdf)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/",
    glob='**/*.pdf',
    loader_cls=PyPDFLoader
)
documents = dir_loader.load()
documents

/tmp/ipykernel_16661/2777653418.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
/home/ayush/Desktop/projects/rag-from-scratch/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'LibreOffice 24.2.7.2', 'creator': 'LibreOffice 24.2.7.2', 'creationdate': '2026-09-14T22:13:49+05:30', 'title': 'Untitled 1', 'source': '../data/training_file.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='The three-body problem is a classical mechanics challenge that determines the motion of three \ncelestial bodies moving under their mutual gravitational attraction. Unlike the predictable two-body \nproblem (such as Earth orbiting the Sun), the three-body problem has no general analytical, closed-\nform solution.\nWhy It Has No Exact Solution\n• Non-Linear Complexity: Adding a third gravitational force creates interdependent pulls \nthat prevent simple mathematical integration.\n• The Butterfly Effect: French mathematician Henri Poincaré proved that the system is \ninherently chaotic. Infinitesimally small differences in starting positions or velocities lead to\nradically different long-term trajectories, making exact long-term f

In [2]:
# creating chunks from text loaded

from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_and_chunk(documents,chunk_size=250,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )


    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} into {len(split_docs)} chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# chunks

split 2 into 18 chunks.


In [3]:
# embedding
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, PreTrainedModel
import numpy as np
from typing import List, Any

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded with embedding dimentions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) ->np.ndarray:
        if not self.model:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

loading embedding manager: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2077.30it/s]


model loaded with embedding dimentions: 384


In [10]:
# vector store
import os
import chromadb
import hashlib
import numpy as np

class VectorManager:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._load_store()

    def _load_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name,
            metadata={"description": "pdf documents for rag"})
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error loading vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("no of documents does not match no of embedding")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Deterministic ID: hash of source + page + chunk index
            source = doc.metadata.get('source', 'unknown')
            page = doc.metadata.get('page', 0)
            hash_input = f"{source}_{page}_{i}".encode()
            doc_id = hashlib.md5(hash_input).hexdigest()[:12]
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.upsert(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"added {len(documents)} documents into vector store")
            print(f"total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

vector_store=VectorManager()
vector_store

vector store initialized, collection: pdf_documents
existing documents in collection: 18


In [12]:
# convert text into embeddings
texts = [doc.page_content for doc in chunks]

# generate embedding
embeddings = embedding_manager.generate_embedding(texts)

# store in vector db
vector_store.add_documents(chunks, embeddings)

generating embedding for 18 texts..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]

generated embeddings with shape: (18, 384)
added 18 documents into vector store
total documents in collection: 18
